# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [3]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

if google_api_key and google_api_key.startswith(("AIz", "AQ.")) and len(google_api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"    
MODEL = 'gemini-3.5-flash-lite'
openai = OpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=google_api_key
)

API key looks good so far


In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have gemini-3.5-flahs-lite figure out which links are relevant

### Use a call to gemini-3.5-flahs-lite to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'social profile', 'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-3.5-flash-lite
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'social media', 'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 9 relevant links


{'links': [{'type': 'company page', 'url': 'https://huggingface.co'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'brand assets', 'url': 'https://huggingface.co/brand'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to gemini-3.5-flash-lite

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/embeddinggemma-2
Updated
7 days ago
•
125k
•
2.02k
jialinyyzz/humanizer
Updated
5 days ago
•
54k
•
1.39k
Qwen/Qwen-Image-2.1-Turbo
Updated
1 day ago
•
9.67k
•
674
abenzerps/Qwen-Image-2.1-Uncensored-Quantized
Updated
1 day ago
•
2.25M
•
4.16k
Cloudflare/clef
Updated
4

In [24]:
#brochure_system_prompt = """
#You are an assistant that analyzes the contents of several relevant pages from a company website
#and creates a short brochure about the company for prospective customers, investors and recruits.
#Respond in markdown without code blocks.
#Include details of company culture, customers and careers/jobs if you have the information.
#"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 7 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/embeddinggemma-2\nUpdated\n7 days ago\n•\n125k\n•\n2.02k\njialinyyzz/humanizer\nUpdated\n5 days ago\n•\n54k\n

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gemini-3.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 8 relevant links


# Welcome to Hugging Face: The AI Community Building the Future

**Hugging Face** is the premier open collaboration platform where the global machine learning community comes together to create, discover, and build the future of artificial intelligence. Recognized as the "Home of Machine Learning," Hugging Face empowers developers, researchers, and enterprises to innovate faster using the leading open-source ML ecosystem.

---

## What We Do

Hugging Face provides an all-in-one platform and suite of developer tools designed to streamline the machine learning lifecycle—from research and dataset creation to model hosting and real-time inference.

### Core Ecosystem Highlights
* **2M+ AI Models:** Access, test, and integrate cutting-edge open-source models across Natural Language Processing (NLP), Computer Vision (CV), Audio, Reinforcement Learning (RL), and multimodal AI.
* **500K+ Datasets:** Explore and share diverse datasets tailored for training, evaluation, and fine-tuning ML models.
* **1M+ Applications (Spaces):** Discover and host interactive ML demos, agents, and apps built using the Hugging Face open-source stack.
* **HuggingChat:** An open-source, conversational AI experience driven by community-hosted models.

---

## Solutions for Enterprises & Developers

Whether you are an independent creator or a global enterprise, Hugging Face offers infrastructure and support designed to scale your AI initiatives securely and efficiently.

* **Inference Endpoints & Providers:** Deploy production-ready models instantly onto managed hardware with low latency.
* **Storage Buckets:** Seamlessly manage and store large ML assets and data.
* **Hugging Face PRO & Enterprise Support:** Unlock premium developer features, dedicated enterprise support, custom privacy controls, and specialized team collaboration solutions.
* **HF Open Source Stack:** Move faster with industry-standard libraries trusted by millions of developers worldwide.

---

## Culture & Community

At Hugging Face, open source is not just a feature—it is at the core of everything we do. Our culture is defined by open collaboration, scientific transparency, and community empowerment.

* **Collaborative & Open-First:** We believe the best AI tools are built together. We actively support open research, ethical AI frameworks, and accessible machine learning.
* **Vibrant Community Hub:** With thriving Discord channels, active developer forums, community blogs, and Daily Papers curation, Hugging Face connects developers with the latest breakthroughs in AI research.
* **Cutting-Edge Research Areas:** From diffusion models and game development to RLHF, robotics (LeRobot), and AI ethics, our ecosystem fosters continuous learning and exploration.

---

## Join the AI Revolution

### For Developers & Researchers
Start collaborating today! Host your models, share datasets, or showcase your latest AI apps to millions of users around the globe.

### For Prospective Recruits
Join a team at the epicenter of the modern AI movement. If you are passionate about open-source software, impactful research, and building tools that empower developers worldwide, Hugging Face is the place for you.

### For Investors & Partners
As the central platform powering modern AI development, Hugging Face provides the core infrastructure driving transformation across industries. 

---

**Explore, build, and innovate with us today:** Visit [Hugging Face](https://huggingface.co) to create a free account, browse trending models, or contact our Enterprise team.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemini-3.5-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 8 relevant links


KeyboardInterrupt: 

In [25]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 8 relevant links


# Welcome to Hugging Face: Because Machines Need Love Too!

## What is This Sorcery?
Forget cold, calculating monoliths. Welcome to **Hugging Face**, the warm, fuzzy, emoji-loving epicentre of the artificial intelligence universe! We are the ultimate digital clubhouse where machine learning wizards, data nerds, and sci-fi dreamers unite to build the future. 

Whether you want to generate a cinematic masterpiece from a simple text prompt, swap your face with an uncensored anime character, or browse through over **2 million models**, 500k+ datasets, and a million AI apps—we’ve got you covered. Consider us GitHub, but with way more personality and significantly more robots trying to write poetry.

---

## Why Our Customers Heart Us
Our platform is built for anyone who wants to move fast and break... well, preferably *not* production, but definitely barriers. 
* **Data Hoarders & Model Nerds:** With unlimited public models, datasets, and storage buckets, you can play scientist to your heart's content.
* **Enterprise Overlords:** Looking to bring AI into the boardroom? Our Team & Enterprise plans, inference endpoints, and enterprise support ensure your corporate algorithms play nicely together without summoning a rogue sentient toaster.

---

## Culture: Math, Mayhem, & Emojis
Our company culture is a delicate ecosystem powered by pure caffeine, endless math symbols ($\sum, \lambda, \nabla, \infty$), and a healthy obsession with open-source collaboration. We believe that building the future shouldn't feel like a corporate tax audit. We chat on Discord, debate daily papers, and run wild experiments on Zero-powered spaces. If your idea of a good time involves arguing over the optimal transformer architecture while wearing a silly hat, you'll fit right in.

---

## Come Work With Us! (Jobs & Careers)
Do you dream in Python? Can you talk dirty to a neural network? Are you capable of maintaining a straight face while explaining why "Best Face Swap AI" is critical to national security? 

We are always looking for brilliant minds to join our band of digital revolutionaries. At Hugging Face, your career will scale faster than a trending model on our leaderboard. Come for the cutting-edge machine learning infrastructure; stay because we actually hug back (metaphorically speaking, through code).

**Join us today. The robots are waiting, and they want a hug.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>